[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_on_Computational_Physics/blob/main/2.Program_structure/Semana_02_EstructurasControl_comentado_1_revisado.ipynb)

# Semana 2 — Estructuras de Control, Ciclos y Condicionales (C³)
**Física Computacional — 106018C** · Lenguaje: **Python**

Este notebook es la parte práctica de la Semana. La teoría (condicionales, criterios de parada, representación IEEE 754, precisión de máquina) está en la presentación `Semana_02_EstructurasControl.pdf` — aquí la ponemos a correr, la comprobamos con código real, y cerramos con el **Taller 2: suma de series**.

## Objetivos de la sesión

- Repasar condicionales y ciclos en Python (`if`/`elif`/`else`, `for`, `while`).
- Diseñar y aplicar criterios de parada en algoritmos iterativos.
- Explorar cómo Python representa números grandes y pequeños (y dónde sí puede desbordarse).
- Codificar y decodificar números en formato IEEE 754 con `struct`.
- Medir experimentalmente la precisión de máquina (ε_m) y verificarla contra `numpy.finfo`.
- Entender el **condicionamiento** de un problema (número de condición) y ver error de propagación y pérdida de significancia — incluyendo cancelación catastrófica — con ejemplos que se rompen de verdad, desde series numéricas hasta un cálculo de física real.
- Resolver el Taller 2: sumar la serie de Taylor de $\sin(x)$ con un criterio de parada por tolerancia.

## 1. Condicionales y ciclos en Python

Rápido antes de avanzar — si esto ya te resulta familiar de tu curso anterior o de la clase,sigue de largo al Ejercicio 1.

In [ ]:
# Deficion de funcion para clasificacion
def clasificar_temperatura(temp):  # Define una función llamada clasificar_temperatura que recibe la variable temp.
    if temp > 100:                 # Comprueba si la temperatura es mayor que 100.
        return "Ebullicion"        # Si la condición anterior es verdadera, devuelve el texto "Ebullicion".
    elif temp == 0:                # Si no se cumplió la condición anterior, comprueba si la temperatura es exactamente 0.
        return "Congelacion"       # Si temp es igual a 0, devuelve el texto "Congelacion".
    else:                          # Si ninguna de las condiciones anteriores se cumple, ejecuta este bloque.
        return "Liquido"           # Devuelve el texto "Liquido".

for t in [25, 0, 150]:             # Recorre, uno por uno, los valores 25, 0 y 150; cada valor se guarda temporalmente en t.
    print(t, "->", clasificar_temperatura(t))  # Muestra t y el resultado de llamar la función con ese valor.


### Ejercicio 1 — Arreglar el código

La siguiente función tiene un error de sintaxis. Corrígelo y vuelve a correr la celda.

In [ ]:
def clasificar_temperatura_bug(temp):  # Define una función similar a la anterior, pero contiene intencionalmente un error de sintaxis.
    if temp > 100                      # ERROR INTENCIONAL: falta el signo ":" al final de la condición if.
        return "Ebullicion"            # Esta línea solo podría ejecutarse si la sintaxis del if fuera correcta.
    elif temp == 0:                    # Comprueba si temp es exactamente igual a 0.
        return "Congelacion"           # Devuelve "Congelacion" cuando temp vale 0.
    else:                              # Se ejecutaría cuando ninguna condición anterior fuera verdadera.
        return "Liquido"               # Devuelve "Liquido" en el caso restante.

print(clasificar_temperatura_bug(25))  # Intenta llamar la función; Python mostrará el error de sintaxis antes de ejecutarla.


<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> en Python la indentación <i>es</i> la estructura del programa (no hay <code>end if</code> ni llaves). Un error de indentación no es solo un problema de estilo — cambia qué líneas pertenecen a qué bloque.
</div>


## 2. Criterios de parada, en la práctica

Un criterio de parada bien diseñado detiene el ciclo cuando el aporte de cada nueva iteración deja de ser significativo — no antes (resultado impreciso) ni mucho después (tiempo de cómputo desperdiciado).

Calentamiento: sumemos la serie geométrica $1 + \tfrac12 + \tfrac14 + \tfrac18 + \cdots$ (converge a 2), deteniéndonos cuando el término por agregar sea menor que una tolerancia.

In [ ]:
def suma_geometrica(tol=1e-6, max_iter=100):  # Define una función con tolerancia 10^-6 y un máximo de 100 iteraciones.
    term = 1.0                                # Inicializa el primer término de la serie geométrica.
    total = 0.0                               # Inicializa en cero la variable que acumulará la suma.
    for i in range(max_iter):                 # Repite el bloque hasta max_iter veces; i toma los valores 0, 1, 2, ...
        total += term                         # Suma el término actual al acumulador; equivale a total = total + term.
        if term < tol:                        # Comprueba si el término actual ya es menor que la tolerancia deseada.
            break                             # Interrumpe inmediatamente el ciclo cuando se cumple el criterio de parada.
        term = term / 2                       # Calcula el siguiente término dividiendo el término actual entre 2.
    return total, i + 1                       # Devuelve la suma obtenida y el número de iteraciones realizadas.

suma, iteraciones = suma_geometrica()  # Ejecuta la función y guarda sus dos resultados en suma e iteraciones.
print(f"suma = {suma}  (en {iteraciones} iteraciones; el valor exacto es 2.0)")  # Imprime los resultados usando una f-string.


<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Cuidado:</b> si el criterio de parada nunca se cumple (por un error de signo, una tolerancia demasiado estricta, o una serie que no converge), el ciclo puede correr para siempre. Por eso <code>suma_geometrica</code> también tiene un <code>max_iter</code> como red de seguridad — <b>todo ciclo <code>while</code> en este curso debe tener una salida garantizada</b>, ya sea por convergencia o por un tope de iteraciones.
</div>


## 3. Números grandes y pequeños en Python

Una particularidad de Python frente a Fortran o C++: **los enteros de Python tienen precisión arbitraria** — no están limitados a 32 o 64 bits, así que en principio no se desbordan (el precio es que las operaciones con enteros gigantes son más lentas).

In [ ]:
grande = 2**1000                                                                     # Calcula 2 elevado a 1000 y guarda el entero resultante en la variable grande.
print("2**1000 tiene", len(str(grande)), "digitos — Python lo maneja sin problema")  # Convierte el entero a texto para contar cuántos dígitos tiene.


Los arreglos de **NumPy**, en cambio, sí usan enteros de ancho fijo (para eficiencia) — y esos sí se desbordan, silenciosamente:

In [ ]:
import numpy as np                               # Importa la biblioteca NumPy y le asigna el alias habitual np.

maximo_int64 = np.int64(9223372036854775807)     # Crea el mayor entero con signo representable por un int64: 2^63 - 1.
with np.errstate(over='ignore'):                 # Abre un bloque donde NumPy ignorará temporalmente las advertencias de overflow.
    resultado = maximo_int64 + np.int64(1)       # Suma 1 al máximo int64 para provocar deliberadamente un desbordamiento.

print("maximo int64        =", maximo_int64)     # Muestra el valor máximo que podía representarse correctamente.
print("maximo int64 + 1     =", resultado, " <- se desbordo y dio la vuelta a negativo")  # Muestra el efecto del overflow.


En CPython, el tipo integrado `float` usa normalmente el formato IEEE 754 de doble precisión (`binary64`) y tiene límites de rango bien definidos:

In [ ]:
import sys                                        # Importa el módulo sys, que permite consultar información del intérprete de Python.

print("Máximo float64:", sys.float_info.max)      # Muestra aproximadamente el mayor valor finito representable por un float de Python.
print("Mínimo normal:", sys.float_info.min)       # Muestra el menor número positivo normalizado representable.
print("Mínimo subnormal aproximado:", 5e-324)     # Muestra un valor cercano al menor subnormal positivo representable.
print("Mínimo subnormal / 2:", 5e-324 / 2, " (underflow -> 0.0)")  # Divide el subnormal entre 2 para ilustrar el underflow hacia cero.
print("1e308 * 10:", 1e308 * 10, " (overflow -> infinito)")        # Multiplica un número enorme para ilustrar el overflow hacia infinito.


## 4. IEEE 754 en la práctica

Repitamos el ejemplo de clase (5.75 → sus 32 bits) y verifiquemos con código en vez de a mano:

In [ ]:
import struct                                    # Importa struct, que permite convertir valores de Python a representaciones binarias de tamaño fijo.

def bits_float32(x):                             # Define una función que devolverá los 32 bits de un número en precisión simple.
    b = struct.pack('>f', x)                     # Convierte x a float de 32 bits; ">f" indica big-endian y precisión simple.
    return ''.join(f'{byte:08b}' for byte in b)  # Convierte cada byte a 8 bits y une los cuatro bytes en una sola cadena.

bits = bits_float32(5.75)  # Convierte el número 5.75 a su representación IEEE 754 de 32 bits.
print("5.75 en IEEE754 (32 bits):", bits)  # Muestra los 32 bits completos.
print(f"  signo={bits[0]}  exponente={bits[1:9]}  mantisa={bits[9:]}")  # Separa y muestra signo, exponente y mantisa.


### Ejercicio 2 — Codifica otro número

Usa `bits_float32` para codificar la velocidad de la luz, `c = 2.99792458e8`. ¿Cuántos bits de exponente esperas que cambien respecto al ejemplo de 5.75?

In [ ]:
# Tu código aquí: este ejercicio reutiliza la función bits_float32 definida en la celda anterior.
c = 2.99792458e8          # Guarda en c la velocidad de la luz en el vacío, expresada en m/s.
print(bits_float32(c))    # Convierte c a float32 y muestra su representación binaria IEEE 754.


## 5. Precisión de máquina (ε_m)

Igual que se mencino en clase, dividimos ε a la mitad hasta que sumarlo a 1 deja de producir un cambio.

### Ejercicio 3 — Completar el código

Completa los dos espacios en blanco (`____`) para que el ciclo encuentre ε_m.

In [ ]:
eps = 1.0                  # Comienza con un valor grande para eps; después se irá reduciendo hasta encontrar la precisión de máquina.
while 1.0 + eps ____ 1.0:  # COMPLETAR: escribe el operador que mantiene el ciclo mientras 1 + eps todavía sea distinguible de 1.
    eps_anterior = eps     # Guarda el último valor de eps que todavía produjo un cambio detectable.
    eps = eps ____         # COMPLETAR: escribe la operación que reduce eps a la mitad en cada iteración.

print("Epsilon de maquina (experimental):", eps_anterior)  # Muestra la última separación distinguible encontrada experimentalmente.


In [ ]:
import numpy as np                    # Importa NumPy para consultar directamente las propiedades del tipo float64.
print("numpy.finfo(float64).eps:      ", np.finfo(np.float64).eps)  # Muestra el epsilon de máquina reportado por NumPy para float64.
print("coincide con tu resultado:", eps_anterior == float(np.finfo(np.float64).eps))  # Compara el valor experimental con el de NumPy.


## 6. Condicionamiento y error de propagación: cuando el problema mismo amplifica el error

Hasta ahora hemos visto *cómo* aparece el error (redondeo, truncamiento, criterios de parada). Falta una pregunta distinta: ¿qué tan sensible es el *problema* mismo a pequeños cambios en la entrada? Esa sensibilidad se llama **condicionamiento**, y determina si incluso un algoritmo perfecto puede terminar dando un resultado inútil.

### (a) Número de condición

Si queremos evaluar $y = f(x)$ pero en realidad calculamos con una entrada ligeramente distinta $\hat{x} = x + \Delta x$, el **número de condición** mide cuánto se amplifica ese error:

$$\text{cond}(f) \approx \left| \frac{x\,f'(x)}{f(x)} \right| = \frac{|\Delta y / y|}{|\Delta x / x|}$$

- $\text{cond}(f) \approx 1$: el problema es **bien condicionado** — un cambio pequeño en $x$ produce un cambio comparable en $y$.
- $\text{cond}(f) \gg 1$: el problema es **mal condicionado** — un cambio minúsculo en $x$ puede producir un cambio enorme en $y$, sin que el algoritmo tenga ninguna culpa.

Probemos esto con $f(x) = \tan(x)$ cerca de $x = \pi/2$, donde la tangente diverge:

In [ ]:
import math                                                       # Ya lo habíamos importado en la sección 7, pero lo dejamos explícito aquí.

x1 = 1.57079                                                      # Un valor muy cercano a pi/2 (~1.570796...).
x2 = 1.57078                                                      # Un segundo valor a solo 0.00001 de distancia del primero.

y1 = math.tan(x1)                                                 # Evalúa tan(x1).
y2 = math.tan(x2)                                                 # Evalúa tan(x2), con una entrada casi idéntica a x1.

cambio_relativo_entrada = abs(x1 - x2) / abs(x1)                  # Mide qué tan distintas son las entradas, en términos relativos.
cambio_relativo_salida = abs(y1 - y2) / abs(y1)                   # Mide qué tan distintas son las salidas, en términos relativos.

print(f"tan({x1}) = {y1:.5f}")
print(f"tan({x2}) = {y2:.5f}")
print(f"cambio relativo en la entrada: {cambio_relativo_entrada:.2e}")
print(f"cambio relativo en la salida:  {cambio_relativo_salida:.2e}")
print(f"cond(tan) estimado en x1:      {cambio_relativo_salida / cambio_relativo_entrada:.2e}")  # cond = amplificación entre entrada y salida.


<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Cuidado:</b> un número de condición gigante no es un error de programación — es una propiedad del <i>problema matemático</i>, no del código. Ningún algoritmo, por más cuidadoso que sea, puede arreglar un problema mal condicionado; a lo sumo puede evitar empeorarlo (eso es lo que en la sección 7 llamaremos <i>estabilidad</i> del algoritmo, cuando reduzcamos el argumento de <code>sin(x)</code> antes de sumar la serie).
</div>

### (b) Precisión simple vs doble

Un ejemplo clásico, $7 + 10^{-7}$, en Python solo se puede reproducir forzando precisión simple con NumPy — el `float` nativo de Python ya es precisión doble:

In [ ]:
print("float64 (nativo):", 7.0 + 1e-7, " -> cambio?", (7.0 + 1e-7) != 7.0)        # Comprueba si sumar 10^-7 cambia un número almacenado como float64.
print("float32 (numpy):  ", np.float32(7.0) + np.float32(1e-7), " -> cambio?", (np.float32(7.0) + np.float32(1e-7)) != np.float32(7.0))  # Repite la comparación usando precisión simple de NumPy.


<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Cuidado:</b> en precisión simple, <code>7.0 + 1e-7</code> da exactamente <code>7.0</code> — el resultado no cambió, aunque matemáticamente debería. La diferencia (1e-7) es más pequeña que la resolución del <code>float32</code> cerca de 7.
</div>


### (c) Cancelación catastrófica

Restar dos números casi iguales puede destruir casi todos los dígitos significativos:

In [ ]:
resultado = (1e16 + 1) - 1e16  # Realiza una operación donde el +1 puede perderse por la precisión finita al trabajar cerca de 10^16.
print("(1e16 + 1) - 1e16 =", resultado, " (matematicamente deberia ser exactamente 1.0)")  # Compara el resultado computacional con el resultado matemático exacto.


<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> evita restar dos números de magnitud muy similar cuando el resultado te importa numéricamente; si es inevitable, reformula la expresión algebraicamente para evitar la resta directa (por ejemplo, usando una identidad matemática equivalente).
</div>


### (d) Cancelación en un contexto físico: energía total de un átomo de helio

La cancelación no es solo un problema de juguete con números grandes — aparece en cálculos físicos reales. La energía total de un átomo de helio es la suma de su energía cinética y su energía potencial, calculadas por separado y con signos opuestos:

$$E_{\text{total}} = E_{\text{cinetica}} + E_{\text{potencial}}$$

Los siguientes son valores (en unidades atómicas) de sucesivas mejoras históricas en el cálculo de estas dos cantidades por separado (Heath, *Scientific Computing*, cap. 1):

In [ ]:
anio = [1971, 1977, 1980, 1985, 1988]                   # Años de publicación de cálculos sucesivos, cada vez más refinados.
cinetica  = [13.0, 12.76, 12.22, 12.28, 12.40]          # Energía cinética calculada en cada año.
potencial = [-14.0, -14.02, -14.35, -14.65, -14.84]     # Energía potencial calculada en cada año (signo opuesto a la cinética).

print(f"{'año':>6} {'cinetica':>10} {'potencial':>10} {'total':>10}")  # Encabezado de la tabla.
totales = []                                            # Lista donde guardaremos la energía total de cada año.
for a, k, p in zip(anio, cinetica, potencial):          # Recorre los tres arreglos en paralelo, un año a la vez.
    total = k + p                                       # Suma cinética y potencial: aquí ocurre la cancelación.
    totales.append(total)                               # Guarda el total de este año para comparar después.
    print(f"{a:>6} {k:>10.2f} {p:>10.2f} {total:>10.2f}")  # Imprime la fila de la tabla.

cambio_componentes = abs(cinetica[-1] - cinetica[0]) / abs(cinetica[0])  # Cuánto cambió la cinética entre 1971 y 1988, en relativo.
cambio_total = abs(totales[-1] - totales[0]) / abs(totales[0])          # Cuánto cambió el total entre 1971 y 1988, en relativo.
print(f"\ncambio relativo en cinetica: {cambio_componentes:.1%}")
print(f"cambio relativo en el total:  {cambio_total:.1%}   <- amplificado por cancelación")


<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> cuando dos cantidades que se suman con signos opuestos provienen de mediciones o cálculos independientes, sus errores <i>no se cancelan</i> — solo se cancelan sus valores. El error relativo del resultado puede ser mucho mayor que el de cualquiera de los términos por separado. Si un resultado físico depende de una resta (o suma con signos opuestos) de cantidades similares, repórtalo con cautela.
</div>

### (e) La fórmula cuadrática: cancelación con solución conocida

Las dos raíces de $ax^2 + bx + c = 0$ se calculan con

$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

Cuando $b^2 \gg 4ac$ (es decir, $\sqrt{b^2-4ac} \approx |b|$), una de las dos raíces resta dos números casi iguales ($-b$ y $\pm\sqrt{b^2-4ac}$) — cancelación catastrófica. La solución es una identidad algebraica equivalente que evita esa resta:

$$x = \frac{2c}{-b \mp \sqrt{b^2 - 4ac}}$$

In [ ]:
def raices_naive(a, b, c):                                      # Implementación directa de la fórmula cuadrática clásica.
    disc = math.sqrt(b**2 - 4*a*c)                              # Calcula el discriminante (asumimos que es no negativo).
    x1 = (-b + disc) / (2*a)                                    # Primera raíz: aquí ocurre la cancelación si b > 0.
    x2 = (-b - disc) / (2*a)                                    # Segunda raíz: aquí NO hay cancelación si b > 0 (los signos se refuerzan).
    return x1, x2

def raices_estables(a, b, c):                                   # Implementación que evita la cancelación usando la fórmula alternativa.
    disc = math.sqrt(b**2 - 4*a*c)                              # Mismo discriminante que antes.
    if b >= 0:                                                  # Si b es positivo, -b+disc cancela: usamos la fórmula alternativa para x1.
        x1 = (2*c) / (-b - disc)                                # Raíz calculada sin restar cantidades casi iguales.
        x2 = (-b - disc) / (2*a)                                # Esta raíz ya era estable con la fórmula clásica.
    else:                                                       # Si b es negativo, es -b-disc quien cancela: aplicamos la simetría del caso.
        x1 = (-b + disc) / (2*a)                                # Esta raíz ya era estable con la fórmula clásica.
        x2 = (2*c) / (-b + disc)                                # Raíz calculada sin restar cantidades casi iguales.
    return x1, x2

# Caso donde b es enorme frente a a y c: b^2 >> 4ac
a, b, c = 1.0, 1e8, 1.0                                         # Coeficientes diseñados para que -b y sqrt(b^2-4ac) casi se cancelen.

x1_naive, x2_naive = raices_naive(a, b, c)                      # Raíces con la fórmula clásica (una de las dos pierde precisión).
x1_estable, x2_estable = raices_estables(a, b, c)               # Raíces con la fórmula que evita la cancelación.

# La raíz pequeña exacta es aproximadamente -c/b (por Vieta: x1*x2 = c/a), útil como referencia:
referencia_raiz_pequena = -c / b                                # Aproximación de alta precisión de la raíz pequeña, para comparar.

print(f"raiz pequeña (naive):    {x1_naive:.15e}")
print(f"raiz pequeña (estable):  {x1_estable:.15e}")
print(f"referencia (-c/b):       {referencia_raiz_pequena:.15e}")


<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Cuidado:</b> corra la celda anterior y compare <code>raiz pequeña (naive)</code> contra la referencia — la versión clásica puede perder varios dígitos significativos (o incluso dar 0.0) mientras la versión estable coincide con la referencia hasta la precisión de máquina. Igual que con la serie de <code>sin(x)</code> en la sección 7, el algoritmo <i>sí</i> importa, aunque las dos fórmulas sean matemáticamente idénticas.
</div>

## 7. Series e Iteraciones

### A. Suma de la serie de Taylor de $\sin(x)$

$$\sin(x) = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \frac{x^7}{7!} + \cdots = \sum_{n=0}^{\infty} (-1)^n\frac{x^{2n+1}}{(2n+1)!}$$

**Objetivo:** calcular $\sin(x)$ sumando términos de la serie, deteniéndose cuando el **aporte del último término** sea menor que una tolerancia prescrita.

En vez de calcular cada término desde cero (factoriales y potencias grandes, caro y riesgoso), relacionamos cada término con el anterior. Si $t_0=x$, entonces:

$$t_n = -\,t_{n-1}\frac{x^2}{(2n)(2n+1)}, \qquad n=1,2,\ldots$$

### B. Completar el algoritmo

Completa los dos espacios en blanco: la fórmula recursiva del término y el criterio de parada. Para evitar problemas cuando la suma está cerca de cero, usa un criterio mixto del tipo

$$|t_n| < \mathrm{tol}\,\max(1,|S_n|).$$


In [ ]:
def suma_seno(x, tol=1e-8, max_iter=200):            # Define una función para aproximar sin(x) mediante su serie de Taylor.
    term = x                                         # Inicializa el primer término de la serie: t_0 = x.
    total = x                                        # Inicializa la suma parcial con el primer término.
    n = 0                                            # Inicializa el índice usado para construir los términos siguientes.

    for i in range(max_iter):                        # Repite como máximo max_iter veces para evitar un ciclo infinito.
        n += 1                                       # Incrementa n antes de calcular el siguiente término de la serie.
        term = ____                                  # COMPLETAR: usa la relación recursiva para calcular el nuevo término a partir del anterior.
        total = total + term                         # Añade el nuevo término a la suma parcial.

        if ____:                                     # COMPLETAR: escribe el criterio de parada mixto basado en term, tol y total.
            break                                    # Sale del ciclo cuando el último término ya es suficientemente pequeño.

    return total, i + 1  # Devuelve la aproximación calculada y el número de iteraciones realizadas.


**Prueba tu implementación** contra `math.sin` para varios valores de $x$, y arma la tabla que pide el taller: $x$, número de iteraciones, suma, y error relativo respecto al valor exacto.

In [ ]:
import math  # Importa el módulo math para usar la función seno de referencia.

print(f"{'x':>6} {'imax':>6} {'suma':>16} {'error relativo':>16}")  # Imprime el encabezado de una tabla con columnas alineadas.
for x in [0.5, 1.0, 2.0, 3.0, 6.0]:  # Recorre varios valores de x para probar el algoritmo.
    approx, iters = suma_seno(x)  # Calcula la aproximación de sin(x) y guarda también las iteraciones utilizadas.
    exacto = math.sin(x)  # Calcula el valor de referencia usando la implementación de math.sin.
    error_relativo = abs(approx - exacto) / abs(exacto)  # Calcula el error relativo de la aproximación respecto al valor de referencia.
    print(f"{x:6.2f} {iters:6d} {approx:16.10f} {error_relativo:16.2e}")  # Imprime una fila de la tabla con formato numérico.


### ¿Y para $x$ grande?

Probemos con $x = 50$ — mucho más allá del rango donde la serie converge rápido.

In [ ]:
approx, iters = suma_seno(50.0)  # Aplica directamente la serie de Taylor a un argumento grande, x = 50.
exacto = math.sin(50.0)  # Calcula el seno de referencia con math.sin para poder comparar.
print(f"x=50:  aproximado={approx}   exacto={exacto}   iteraciones={iters}")  # Muestra la aproximación, el valor de referencia y el costo en iteraciones.


<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Cuidado:</b> para x grande, la suma se dispara a valores absurdos (miles, en vez de estar entre -1 y 1) — no es que el criterio de parada esté mal escrito: los términos intermedios se vuelven enormes antes de empezar a cancelarse, y la precisión finita de <code>float</code> no logra representarlos con suficiente exactitud como para que la cancelación funcione. Es error de propagación puro, el mismo fenómeno de la sección 6, pero acumulado durante docenas de iteraciones.
</div>


**La solución:** usar la identidad $\sin(x + 2n\pi) = \sin(x)$ para reducir cualquier $x$ a un valor entre $-\pi$ y $\pi$ antes de sumar la serie.

In [ ]:
def suma_seno_reducida(x, tol=1e-8, max_iter=200):  # Define una versión que primero reduce el argumento antes de evaluar la serie.
    x_reducido = math.remainder(x, 2 * math.pi)  # Reduce x módulo 2*pi a un intervalo centrado aproximadamente en [-pi, pi].
    return suma_seno(x_reducido, tol, max_iter)  # Evalúa la serie usando el argumento reducido y devuelve el resultado.

for x in [50.0, 100.0, -237.0]:  # Prueba la función con varios argumentos grandes positivos y negativos.
    approx, iters = suma_seno_reducida(x)  # Calcula sin(x) usando reducción de argumento y la serie de Taylor.
    exacto = math.sin(x)  # Obtiene el valor de referencia con math.sin.
    print(f"x={x:8.1f}   aproximado={approx: .10f}   exacto={exacto: .10f}   iteraciones={iters}")  # Compara resultados e informa las iteraciones.


<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> cuando un algoritmo numérico falla solo en un rango de valores de entrada, no aumentes a ciegas <code>max_iter</code> ni relajes la tolerancia — primero entiende <i>por qué</i> falla (aquí, precisión perdida por cancelación en términos grandes) y corrige la causa (aquí, reducir el rango de entrada con una identidad matemática).
</div>


## 8. Buenas prácticas de programación — síntesis

1. En Python la indentación es estructura, no estilo — revísala con cuidado (sección 1).
2. Todo ciclo `while` necesita una salida garantizada: convergencia *y* un tope de iteraciones de respaldo (sección 2).
3. Los `int` de Python no se desbordan; los enteros de NumPy sí — no asumas que todo en Python tiene precisión arbitraria (sección 3).
4. No uses `==` para verificar igualdad de resultados numéricos aproximados esperando exactitud matemática perfecta — la precisión finita casi nunca lo permite (secciones 5-6).
5. Evita restar cantidades de magnitud casi igual cuando el resultado importa numéricamente; si es inevitable, busca una identidad algebraica equivalente que evite la resta directa (fórmula cuadrática, sección 6).
6. Ante un algoritmo que diverge en cierto rango, diagnostica la causa antes de "parchar" con más iteraciones o tolerancias más flexibles (sección 7).
7. Antes de optimizar un algoritmo, pregúntate si el *problema* mismo está mal condicionado ($\text{cond}(f) \gg 1$) — eso no lo arregla ningún algoritmo, solo cambiar el problema o aumentar la precisión (sección 6).

## 9. Hoja de referencia rápida

| Concepto | Sintaxis en Python |
|---|---|
| Condicional | `if condicion:` … `elif otra:` … `else:` |
| Ciclo contable | `for i in range(n):` |
| Ciclo condicional | `while condicion:` |
| Comparación | `==  !=  <  <=  >  >=` |
| Lógicos | `and  or  not` |
| Codificar IEEE754 | `struct.pack('>f', x)` |
| Épsilon de máquina | `numpy.finfo(float).eps` |
| Reducir ángulo | `math.remainder(x, 2*math.pi)` |
| Número de condición (aprox.) | `abs(x * fprima(x) / f(x))` |

## 10. Taller 2

### A. Jugando con la precision en Python
1. Complete y depure `suma_seno` (sección 7, "B. Completar el algoritmo") si aún no lo ha hecho.
2. Genere la tabla de convergencia (como en la sección 7) para al menos 6 valores de $x$ entre $0$ y $4\pi$, incluyendo el número de iteraciones, la suma y el error relativo frente a `math.sin`.
3. Con los datos de la tabla del punto anterior, compare el error relativo de cada caso contra la tolerancia esperada $10^{-8}$: ¿en todos los valores de $x$ que probó el error relativo queda por debajo de esa tolerancia? Si en algún caso no es así, indique para cuáles valores de $x$ ocurre y explique a qué se debe.
4. Repita el experimento de la sección 7 para $x$ grande (por ejemplo, $x=100$) **sin** la reducción de periodicidad, y explique en una o dos frases qué está pasando numéricamente.
5. Compare su implementación recursiva con una versión "ingenua" que calcule cada término desde cero usando `math.factorial` y `x**potencia` — ¿en qué valores de $x$ empieza a notarse la diferencia de desempeño o de precisión?

### B. Decaimiento radiactivo con Python (Entregar Reporte)

Durante la semana anterior estudiamos el **decaimiento radiactivo utilizando FORTRAN**. En este ejercicio retomaremos el mismo sistema físico, pero utilizaremos Python para trabajar con un conjunto de valores, analizar los resultados y representarlos gráficamente.

El número de núcleos radiactivos presentes en una muestra evoluciona de acuerdo con

$$
N(t)=N_0e^{-\lambda t},
$$

donde $N_0$ es el número inicial de núcleos y $\lambda$ es la constante de decaimiento. Esta constante se relaciona con la vida media $t_{1/2}$ mediante

$$
\lambda=\frac{\ln(2)}{t_{1/2}}.
$$

Considere una muestra con

$$
N_0=10\,000
$$

núcleos radiactivos y una vida media de

$$
t_{1/2}=5\ \mathrm{años}.
$$

#### Actividades

1. Calcule la constante de decaimiento $\lambda$.

2. Utilice `numpy.linspace()` para crear un arreglo de tiempos entre **0 y 30 años**. Utilice un número suficiente de puntos para observar claramente la evolución de la muestra.

3. Calcule $N(t)$ para todos los valores del arreglo de tiempos utilizando las operaciones vectorizadas de NumPy.

   **No utilice un ciclo `for` para realizar este cálculo.**

4. Utilice `matplotlib` para representar gráficamente $N(t)$ en función del tiempo. La gráfica debe incluir:

   - título;
   - nombre de los ejes;
   - unidades;
   - una cuadrícula que facilite la lectura de los resultados.

5. Identifique sobre la gráfica los tiempos correspondientes a **una, dos y tres vidas medias**. Observe qué fracción de la muestra permanece después de cada una de ellas.

6. Determine aproximadamente el tiempo en el cual permanece:

   - el **50 %** de la muestra inicial;
   - el **25 %** de la muestra inicial;
   - el **10 %** de la muestra inicial.

**Objetivo del ejercicio:** aplicar las herramientas básicas de Python estudiadas esta semana a un problema físico conocido y comenzar a utilizar Python para el **manejo, análisis y visualización de resultados científicos**.

### C. Celda de verificación (PASS/FAIL)

Para agilizar la revisión, la tarea debe cerrar con una celda que compruebe automáticamente 2-3 casos conocidos e imprima PASS/FAIL — no hace falta rastrear la lógica a mano para calificar. Abajo hay una celda de verificación para la Parte A (usa `suma_seno_reducida`, ya definida en la sección 7) y una plantilla para la Parte B: complete `N_decaimiento` con $N(t) = N_0 e^{-\lambda t}$ y la celda le dirá si sus resultados coinciden con los valores esperados (recordando el caso de la vida media del carbono-14 de la Semana 1: al cabo de una vida media siempre queda el 50 % de la muestra).

In [ ]:
# --- Verificación Parte A: suma_seno_reducida ---
casos_a = [                                        # Lista de (x, valor_esperado, tolerancia) para probar la Parte A.
    (0.5, math.sin(0.5), 1e-8),                    # Caso dentro del rango normal de convergencia rápida.
    (100.0, math.sin(100.0), 1e-6),                # Caso con x grande: solo debe pasar si se usó reducción de argumento.
    (-237.0, math.sin(-237.0), 1e-6),              # Caso con x grande y negativo.
]

for x, esperado, tol in casos_a:                   # Recorre cada caso de prueba uno por uno.
    obtenido, _ = suma_seno_reducida(x)             # Calcula sin(x) con la función que el estudiante debe haber completado.
    ok = abs(obtenido - esperado) < tol             # Compara contra el valor de referencia dentro de la tolerancia.
    estado = "PASS" if ok else "FAIL"               # Traduce el resultado booleano a texto legible.
    print(f"{estado}  x={x:>8.1f}   obtenido={obtenido: .10f}   esperado={esperado: .10f}")  # Imprime el veredicto de este caso.


In [ ]:
# --- Parte B: complete N_decaimiento y luego corra la verificación ---
def N_decaimiento(t, N0=10000, t_half=5.0):        # Complete esta función según N(t) = N0 * exp(-lambda*t).
    raise NotImplementedError("Complete N_decaimiento antes de correr la verificación")  # Quite esta línea al completar la función.

casos_b = [                                        # Lista de (t, fracción_esperada): fracción de N0 que debería quedar en cada instante.
    (0.0, 1.00),                                   # En t=0 debe quedar el 100% de la muestra.
    (5.0, 0.50),                                   # En t = t_1/2 (una vida media) debe quedar el 50%.
    (10.0, 0.25),                                  # En t = 2*t_1/2 (dos vidas medias) debe quedar el 25%.
]

for t, fraccion_esperada in casos_b:               # Recorre cada caso de prueba.
    try:                                           # Evita que un NotImplementedError detenga toda la celda de verificación.
        obtenido = N_decaimiento(t) / 10000        # Calcula qué fracción de N0 queda en el instante t.
        ok = abs(obtenido - fraccion_esperada) < 1e-3  # Compara contra la fracción esperada con tolerancia razonable.
        estado = "PASS" if ok else "FAIL"
        print(f"{estado}  t={t:>5.1f} años   fraccion={obtenido:.3f}   esperada={fraccion_esperada:.3f}")
    except NotImplementedError as e:               # Si aún no se ha completado la función, lo informa sin detener la celda.
        print(f"FAIL  t={t:>5.1f} años   ({e})")


## 11. Referencias

- Presentación de la Semana 2: `Semana_02_EstructurasControl.pdf`.
- Heath, M. T. *Scientific Computing: An Introductory Survey*, 2.ª ed. revisada — Cap. 1 (*Scientific Computing*): fuente de los ejemplos de condicionamiento (número de condición), cancelación en el átomo de helio y la fórmula cuadrática de la sección 6.